In [ ]:
import pandas as pd

In [45]:
hacker_df = pd.read_json("data/hacker_news_data.json")

In [46]:
hacker_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 609976 entries, 0 to 609975
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            609976 non-null  int64  
 1   title         609976 non-null  object 
 2   url           580038 non-null  object 
 3   points        609976 non-null  int64  
 4   num_comments  609975 non-null  float64
 5   author        609976 non-null  object 
 6   created_at_i  609976 non-null  int64  
 7   tags          609976 non-null  object 
dtypes: float64(1), int64(3), object(4)
memory usage: 37.2+ MB


In [69]:
hacker_df[hacker_df["points"] > 100]

,id,title,url,points,num_comments,author,created_at_i,tags,datetime_utc
15,42570138,Are there more trees on Earth than stars in th...,https://www.snopes.com/fact-check/trees-stars-...,108,90.0,alexmolas,1735773653,"[story, author_alexmolas, story_42570138]",2025-01-01 23:20:53+00:00
35,42569913,My 25-year adventure in AI and ML,https://austinhenley.com/blog/25yearsofai.html,194,37.0,ibobev,1735771276,"[story, author_ibobev, story_42569913]",2025-01-01 22:41:16+00:00
40,42569880,Counterculture legend who invented bell-bottom...,https://www.sfgate.com/local/article/peggy-cas...,122,41.0,anigbrowl,1735770941,"[story, author_anigbrowl, story_42569880]",2025-01-01 22:35:41+00:00
89,42569236,Rails for everything,https://literallythevoid.com/blog/rails_for_ev...,344,243.0,FigurativeVoid,1735765084,"[story, author_FigurativeVoid, story_42569236]",2025-01-01 20:58:04+00:00
101,42569109,My favourite computer ergonomics hack,https://blog.jacobvosmaer.nl/0036-beeper/,236,71.0,vortex_ape,1735763993,"[story, author_vortex_ape, story_42569109]",2025-01-01 20:39:53+00:00
...,...,...,...,...,...,...,...,...,...
609670,50256985,Roblox downloaded 145GB models from us (includ...,https://open3dlab.com/news/roblox-downloaded-1...,486,281.0,choult,1793096870,"[story, author_choult, story_50256985]",2026-10-27 10:27:50+00:00
609673,50256952,Gforth 1.0,https://gforth.org/manual/1.0/NEWS.html,173,26.0,joshcsimmons,1793096601,"[story, author_joshcsimmons, story_50256952]",2026-10-27 10:23:21+00:00
609811,50255298,The Mythical Man-Month in the Age of AI,https://j11g.com/the-mythical-man-month-in-the...,500,364.0,ilreb,1793082337,"[story, author_ilreb, story_50255298]",2026-10-27 06:25:37+00:00
609903,50253939,I still teach OpenGL ES 3.0,https://eliasfarhan.ch/graphics/cpp/2026/01/27...,157,93.0,majorchord,1793067310,"[story, author_majorchord, story_50253939]",2026-10-27 02:15:10+00:00


In [68]:
hacker_df["datetime_utc"] = pd.to_datetime(
    hacker_df["created_at_i"], unit="s", utc=True
)

In [49]:
mask = (hacker_df["url"].notnull()) & (hacker_df["url"].duplicated(keep="first"))

hacker_df = hacker_df.loc[~mask]

In [70]:
hacker_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 530760 entries, 0 to 609975
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype              
---  ------        --------------   -----              
 0   id            530760 non-null  int64              
 1   title         530760 non-null  object             
 2   url           500822 non-null  object             
 3   points        530760 non-null  int64              
 4   num_comments  530760 non-null  float64            
 5   author        530760 non-null  object             
 6   created_at_i  530760 non-null  int64              
 7   tags          530760 non-null  object             
 8   datetime_utc  530760 non-null  datetime64[ns, UTC]
dtypes: datetime64[ns, UTC](1), float64(1), int64(3), object(4)
memory usage: 40.5+ MB


In [64]:
hacker_df.dtypes

id                int64
title            object
url              object
points            int64
num_comments    float64
author           object
created_at_i      int64
tags             object
datetime_utc     object
dtype: object

In [ ]:
hacker_df.groupby(hacker_df["datetime_utc"].dt.to_period("M"))["points"].agg(
    ["mean", "size"]
)

/var/folders/5l/1_fz5gjn0xs8pvv6_ygxmt580000gn/T/ipykernel_53658/2466106257.py:1: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  hacker_df.groupby(hacker_df["datetime_utc"].dt.to_period("M"))["points"].agg(["mean", "size"])


,mean,size
datetime_utc,,
2025-01,18.938777,22704
2025-02,21.162169,19899
2025-03,19.503571,22265
2025-04,19.565095,21246
2025-05,17.993331,22043
2025-06,17.140148,20778
2025-07,18.641796,22141
2025-08,19.946294,22083
2025-09,20.454883,21821


In [ ]:
df = hacker_df.assign(
    month=hacker_df["datetime_utc"].dt.tz_convert(None).dt.to_period("M")
)

monthly = df.groupby("month").agg(
    n=("id", "size"),
    hit_rate=("points", lambda s: (s > 100).mean()),
    low_rate=("points", lambda s: (s <= 2).mean()),
    median_points=("points", "median"),
    authors=("author", "nunique"),
)
monthly["per_day"] = monthly["n"] / monthly.index.days_in_month
monthly["posts_per_author"] = monthly["n"] / monthly["authors"]

In [73]:
monthly

,n,hit_rate,low_rate,median_points,authors,per_day,posts_per_author
month,,,,,,,
2025-01,22704,0.044882,0.527264,2.0,8872,732.387097,2.559062
2025-02,19899,0.050857,0.496005,3.0,8167,710.678571,2.436513
2025-03,22265,0.046036,0.482865,3.0,8727,718.225806,2.551278
2025-04,21246,0.043679,0.440930,3.0,8631,708.200000,2.461592
2025-05,22043,0.043143,0.464909,3.0,8973,711.064516,2.456592
2025-06,20778,0.040861,0.456396,3.0,8566,692.600000,2.425636
2025-07,22141,0.044804,0.435346,3.0,9254,714.225806,2.392587
2025-08,22083,0.049359,0.435584,3.0,9083,712.354839,2.431245
2025-09,21821,0.048302,0.462307,3.0,8867,727.366667,2.460923


In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

df = hacker_df.copy()
df["datetime_utc"] = pd.to_datetime(df["datetime_utc"], utc=True)
df = df.sort_values("datetime_utc").reset_index(drop=True)

df["y"] = (df["points"] > 100).astype(int)
df["title"] = df["title"].fillna("")
df["hour"] = df["datetime_utc"].dt.hour.astype(str)
df["weekday"] = df["datetime_utc"].dt.dayofweek.astype(str)
df["date"] = df["datetime_utc"].dt.date

df["domain"] = (
    df["url"]
    .fillna("")
    .str.extract(r"https?://(?:www\.)?([^/:?#]+)", expand=False)
    .str.lower()
    .fillna("no_url")
)
df["prefix"] = np.select(
    [df["title"].str.startswith("Show HN"), df["title"].str.startswith("Ask HN")],
    ["show_hn", "ask_hn"],
    default="other",
)

cut = lambda s: pd.Timestamp(s, tz="UTC")
ts = df["datetime_utc"]
train = df[ts < cut("2026-07-01")]
val = df[(ts >= cut("2026-07-01")) & (ts < cut("2026-09-01"))]
test = df[ts >= cut("2026-09-01")]

In [84]:
df.head()

,id,title,url,points,num_comments,author,created_at_i,tags,datetime_utc,y,hour,weekday,date,domain,prefix
0,42562819,Understanding intuition of the Two child problem,https://stats.stackexchange.com/questions/6593...,3,0.0,firebaze,1735689667,"[story, author_firebaze, story_42562819]",2025-01-01 00:01:07+00:00,0,0,2,2025-01-01,stats.stackexchange.com,other
1,42562822,Exploring where and how the Federal Government...,https://www.usaspending.gov/explorer,4,3.0,nobodyandproud,1735689684,"[story, author_nobodyandproud, story_42562822]",2025-01-01 00:01:24+00:00,0,0,2,2025-01-01,usaspending.gov,other
2,42562847,Static search trees: faster than binary search,https://curiouscoding.nl/posts/static-search-t...,656,232.0,atombender,1735690083,"[story, author_atombender, story_42562847]",2025-01-01 00:08:03+00:00,1,0,2,2025-01-01,curiouscoding.nl,other
3,42562853,The Linux Kernel Hit a Decade Low in 2024 for ...,https://www.phoronix.com/news/2024-Linux-Git-S...,3,1.0,mikece,1735690161,"[story, author_mikece, story_42562853]",2025-01-01 00:09:21+00:00,0,0,2,2025-01-01,phoronix.com,other
4,42562903,Show HN: Generate Nonsensical Math Papers,https://sylvanfranklin.github.io/nonsense/,2,0.0,wmstack,1735690706,"[story, author_wmstack, story_42562903, show_hn]",2025-01-01 00:18:26+00:00,0,0,2,2025-01-01,sylvanfranklin.github.io,show_hn


In [85]:
def precision_at_k_per_day(frame, scores, k=20):
    tmp = pd.DataFrame(
        {"date": frame["date"].values, "y": frame["y"].values, "s": scores}
    )
    top = tmp.sort_values("s", ascending=False).groupby("date").head(k)
    return top["y"].mean()


def evaluate(name, frame, scores):
    return {
        "model": name,
        "pr_auc": average_precision_score(frame["y"], scores),
        "roc_auc": roc_auc_score(frame["y"], scores),
        "precision_at_k/day": precision_at_k_per_day(frame, scores),
    }

In [ ]:
cat_cols = ["domain", "hour", "weekday", "prefix"]
features = cat_cols + ["title"]


def onehot():
    return OneHotEncoder(min_frequency=100, handle_unknown="infrequent_if_exist")


def make_model(preprocessor):
    return Pipeline(
        [
            ("prep", preprocessor),
            ("clf", LogisticRegression(C=1, solver="liblinear", max_iter=1000)),
        ]
    )


preps = {
    "metadata only": ColumnTransformer([("cat", onehot(), cat_cols)]),
    "title + metadata": ColumnTransformer(
        [
            ("cat", onehot(), cat_cols),
            (
                "word",
                TfidfVectorizer(ngram_range=(1, 2), min_df=5, sublinear_tf=True),
                "title",
            ),
        ]
    ),
}

fitted, results = {}, []
for name, prep in preps.items():
    model = make_model(prep).fit(train[features], train["y"])
    fitted[name] = model
    results.append(evaluate(name, val, model.predict_proba(val[features])[:, 1]))

pd.DataFrame(results).round(4)

,model,pr_auc,roc_auc,precision_at_k/day
0,metadata only,0.0773,0.6577,0.1250
1,title + metadata,0.1193,0.7305,0.1871


In [ ]:
model = fitted["title + metadata"]
names = model.named_steps["prep"].get_feature_names_out()
coef = model.named_steps["clf"].coef_[0]
order = np.argsort(coef)

print("Strongest signals for a hit:")
print(list(names[order[-25:]][::-1]))
print("\nStrongest signals for a miss:")
print(list(names[order[:25]]))

Strongest signals for a hit:
['word__launch hn', 'word__hn who', 'word__has died', 'word__you working', 'word__chat control', 'word__grapheneos', 'word__zig', 'cat__domain_anthropic.com', 'word__droid', 'word__ffmpeg', 'word__qwen3', 'word__is hiring', 'word__be hired', 'word__mistral', 'word__flock', 'word__mozilla', 'word__ice', 'cat__domain_jeffgeerling.com', 'word__6502', 'word__2024', 'word__exfiltrates', 'word__resolved', 'word__servo', 'word__pebble', 'word__lisp']

Strongest signals for a miss:
['word__china', 'word__trump', 'word__chinese', 'word__ai', 'word__musk', 'word__warns', 'word__plan', 'word__generator', 'word__agent', 'word__plans', 'word__russia', 'word__quantum', 'word__india', 'word__review', 'word__crypto', 'word__survey', 'word__could', 'word__released', 'word__tool', 'word__lawsuit', 'word__benchmark', 'word__saas', 'word__platform', 'cat__domain_hackaday.com', 'word__for']
